In [62]:
import pyodbc
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()

# prep

In [63]:
# Data prep
query = f"""

SELECT
  CASE WHEN borrower_id < lender_id THEN borrower_id ELSE lender_id END AS a,
  CASE WHEN borrower_id < lender_id THEN lender_id ELSE borrower_id END AS b
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state
WHERE intragroup = 1
GROUP BY
  CASE WHEN borrower_id < lender_id THEN borrower_id ELSE lender_id END,
  CASE WHEN borrower_id < lender_id THEN lender_id ELSE borrower_id END;


"""

df_edges = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_13224\143588195.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_edges = pd.read_sql_query(query, cnxn)


In [64]:
import pandas as pd

# edges: columns ['a','b']
edges = df_edges.values.tolist()

parent = {}
rank = {}

def find(x):
    if parent.setdefault(x, x) != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(x, y):
    rx, ry = find(x), find(y)
    if rx == ry: return
    rxr, ryr = rank.setdefault(rx,0), rank.setdefault(ry,0)
    if rxr < ryr:
        parent[rx] = ry
    elif rxr > ryr:
        parent[ry] = rx
    else:
        parent[ry] = rx
        rank[rx] = rxr + 1

for a,b in edges:
    union(a,b)

# canonical id = min-LEI per component (stable label)
from collections import defaultdict
comps = defaultdict(list)
for x in parent.keys():
    comps[find(x)].append(x)

canon = {rep: min(members) for rep, members in comps.items()}
mapping = []
for x in parent.keys():
    rep = find(x)
    mapping.append((x, canon[rep]))

df_mapping_cc = pd.DataFrame(mapping, columns=["lei","group_id_synth"]).sort_values(["group_id_synth","lei"])


In [65]:
# Data prep
query = f"""

SELECT security_isin
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state f
WHERE security_type IN ('GOVS', 'FIDE')
AND issuer_esa_sector_riad = 'S1311'
AND business_date >= '2023-01-01' 
AND gnlcoll = 'SPEC'
GROUP BY security_isin
HAVING COUNT(DISTINCT security_type) = 2

"""

govs = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_13224\2035066873.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  govs = pd.read_sql_query(query, cnxn)


In [66]:
unique_govs = tuple(govs['security_isin'])

In [67]:
foreign_bg = ('0W1U67PTV5WY3WYWKD79',
'2138008P9NOMBRMROI73',
'213800A9GT65GAES2V60',
'213800G8QEXN34A2YG53',
'213800GVD8L87R18CT98',
'213800IBT39XQ9C4CP71',
'213800NW35DTWHTMX505',
'213800RZ3GCUXMBGYN59',
'2IGI19DL77OX0HC3ZE78',
'4PQUHN3JPFGFNF3BB653',
'4ZHCHI4KYZG2WVRT8631',
'5299007QVIQ7IO64NX37',
'5493000IQQ05Y25L0V92',
'5493000YPN33HF74SN02',
'54930010P7BUGOECPI58',
'5493002XYZZ0CGQ6CB58',
'54930040QPHHWGT1J432',
'54930050SE0SM7CM2G07',
'5493006PWI2H6PX25403',
'5493007VSMFZCPV1NB83',
'5493008GNQHVI377MY19',
'5493009NLZXZGJDOPC94',
'549300BKWHXYEXPV0328',
'549300FH0WJAPEHTIQ77',
'549300HU9EWFS3CNX640',
'549300KP56LL8NKKFL47',
'549300RSY622D5TQWS42',
'549300SXSTGQY3EA1B18',
'549300WDT1HWUMTUW770',
'571474TGEMMWANRLN572',
'724500AAT1DK36059L16',
'9676007O0UF5YB3QPR03',
'BWS7DNS2Z4NPKPNYKL75',
'HV5W8PGLJ127N2SFSM23'
)

In [68]:
euro_area_bg = ('0IKLU6X1B10WK7X42C15',
'1VUV7VQFKUOQSJ21A208',
'2138001YBO7CJNQOEE37',
'21380027LW8AF6I1WA03',
'2138003Z5ZVN16GFYV70',
'2138004VZX8CSGPTDX68',
'21380073P7J4PAD91E29',
'213800AGKVL18YKQSV51',
'213800BWHAS44J2C1B28',
'213800D4LHBCXXEEE235',
'213800DBQIB6VBNU5C64',
'213800FKGFR7Q2ACLS83',
'213800G63T4ER4MSVR22',
'213800HV6TP2I5A6MW58',
'213800I92TAU7I3FP232',
'213800KGF4EFNUQKAT69',
'213800OOQOSULB37T658',
'213800ZIGVOZ992FNQ85',
'222100D7H9VRJEH7DU25',
'222100M2PU043YB7YQ06',
'23B6332KMR0JIZLJG565',
'2534006G7F7F1TFC9T77',
'2534006HF1L4YF10UD91',
'253400N2R0RF14JQ0060',
'2549002MVYWWDX54IB83',
'259400QHDOZWMJ103294',
'259400YLRTOBISHBVX41',
'2W8N8UU78PMDQKZENC08',
'31570010000000036567',
'315700GBLUBZ50S45F53',
'315700GXRKZ452JF2U13',
'3M5E1GQGKL17HI6CPN30',
'3U8WV1YX2VMUHH7Z1Q21',
'48510000156ESYOBV122',
'52965FONQ5NZKP0WZL45',
'52990002O5KK6XOGJ020',
'52990010C4NK412ZL440',
'5299004TE2DYMKEAM814',
'5299005UJX6K7BQKV086',
'52990080NNXXLC14OC65',
'529900AQBND3S6YJLY83',
'529900C214QOT3ZYD838',
'529900C4RSSBWXBSY931',
'529900E1WHT64CB20277',
'529900FWTU88V844B672',
'529900GGYMNGRQTDOO93',
'529900K16YGKC8BES892',
'529900T32UL0CP1FZA06',
'529900UKZBMDBDZIXD62',
'529900VA5CNBWXAONR25',
'549300298FD7AS4PPU70',
'54930056IRBXK0Q1FP96',
'549300685QG7DJS55M76',
'5493007RT80TMKY7TO30',
'549300ABE4K96QOCEH37',
'549300CQ9NLEHMRCU505',
'549300DV870NBWY5W279',
'549300DY78U4CMKNHE48',
'549300DYPOFMXOR7XM56',
'549300FOF121DSRG5867',
'549300FR956J8UJDWQ78',
'549300GOF5A5DHWJLV03',
'549300GRXFI7D6PNEA68',
'549300L7YCATGO57ZE10',
'549300LYFYVPUCG6SY25',
'549300NC3SZTETC10349',
'549300NEBDPH0ZXIF850',
'549300Z6OB1D4ZUBD145',
'63540061DPCBNMCGRY22',
'635400CE9HHFB55PEY43',
'635400GQWXFJDCQXW612',
'635400LNHEPZBRNB5D58',
'635400LRAHYBRUZCIH13',
'724500A11NIP5HCVF984',
'815600154F8F91CF6B05',
'8156002070DA4DCBFF31',
'81560027D07F9BDB8436',
'81560038903FF9FA8F80',
'815600522538355AE429',
'8156007395B20763EB44',
'8156009F40F6523F7022',
'815600A32DA05F693F24',
'8EFE15WY4PBBKG6GZI21',
'95980020140005184148',
'9598009X93GQBNHF1L85',
'959800T0J9ADL4GYSS41',
'9695000O0HNLFNR0FB30',
'969500DCEVPV6UIYK220',
'969500QLO3GN6GUB4P61',
'FI6C7E5PBUB3F9K43B44',
'GP5DT10VX1QRQUKVBK64',
'J4CP7MHCXR8DAQMKIL78',
'NNVPP80YIZGEY2314M97',
'RRAN7P32P0W0YY4XQW79',
'SI5RG2M0WQQLZCXKRM20'
)

In [69]:
foreign_entity = tuple(df_mapping_cc.loc[df_mapping_cc['group_id_synth'].isin(foreign_bg), 'lei'].unique())
euro_area_entity = tuple(df_mapping_cc.loc[df_mapping_cc['group_id_synth'].isin(euro_area_bg), 'lei'].unique())

# wedge

Euro area entities of foreign groups, one row per entity, bond, day and tenor bucket. Tenor is `short` for open term or up to seven days and `term` beyond, so legs are matched within bucket. Four legs with volume weighted rates in percent, average contractual maturity in days (fixed term trades only) and average haircut in percent. Only trades with a rate.

`chain_intra_to_ccp` is the euro area entity borrowing cash intragroup and lending it in the CCP (bond goes out to the non-euro area entity), `wedge_intra_to_ccp = cleared_lending_rate - intra_borrowing_rate`. `chain_ccp_to_intra` is the euro area entity lending cash intragroup and borrowing it in the CCP (bond comes in), `wedge_ccp_to_intra = intra_lending_rate - cleared_borrowing_rate`. Wedges in basis points, positive when the euro area entity keeps a margin. `carry` pools the two, weighted by matched volume.

In [70]:
query = f"""

SELECT business_date, lender_id as entity_id, security_isin, 
CASE
    WHEN s.lender_country_residence IN ('US') THEN 'US'
    WHEN s.lender_country_residence IN ('GB') THEN 'UK'
    WHEN s.lender_country_residence IN
         ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
      THEN 'Euro Area'
    ELSE 'Other'
  END AS residence_group,
CASE
    WHEN contractual_maturity IS NULL OR contractual_maturity <= 7 THEN 'short'
    ELSE 'term'
  END AS tenor,
sum(nominal_value)/1e9 as cleared_lending,
sum(repo_rate*nominal_value)/sum(nominal_value) as cleared_lending_rate,
avg(contractual_maturity) as cleared_lending_maturity,
avg(haircut) as cleared_lending_haircut
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND lender_id in {foreign_entity}
AND intragroup = 0
AND central_clearing = 'cleared'
AND repo_rate IS NOT NULL
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.issuer_esa_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.security_isin IN {unique_govs})
  )
GROUP BY business_date, lender_id, residence_group, tenor, security_isin
ORDER BY business_date, lender_id, residence_group, tenor, security_isin
  
"""

df_cleared_lending = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_13224\316846021.py:44: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_cleared_lending = pd.read_sql_query(query, cnxn)


In [71]:
query = f"""

SELECT business_date, borrower_id as entity_id, security_isin, 
CASE
    WHEN s.borrower_country_residence IN ('US') THEN 'US'
    WHEN s.borrower_country_residence IN ('GB') THEN 'UK'
    WHEN s.borrower_country_residence IN
         ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
      THEN 'Euro Area'
    ELSE 'Other'
  END AS residence_group,
CASE
    WHEN contractual_maturity IS NULL OR contractual_maturity <= 7 THEN 'short'
    ELSE 'term'
  END AS tenor,
sum(nominal_value)/1e9 as cleared_borrowing,
sum(repo_rate*nominal_value)/sum(nominal_value) as cleared_borrowing_rate,
avg(contractual_maturity) as cleared_borrowing_maturity,
avg(haircut) as cleared_borrowing_haircut
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND borrower_id in {foreign_entity}
AND intragroup = 0
AND central_clearing = 'cleared'
AND repo_rate IS NOT NULL
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.issuer_esa_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.security_isin IN {unique_govs})
  )
GROUP BY business_date, borrower_id, residence_group, tenor, security_isin
ORDER BY business_date, borrower_id, residence_group, tenor, security_isin
  
"""

df_cleared_borrowing = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_13224\89401179.py:44: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_cleared_borrowing = pd.read_sql_query(query, cnxn)


In [72]:
df_cleared = df_cleared_lending.merge(df_cleared_borrowing, on = ['business_date', 'entity_id', 'residence_group', 'tenor', 'security_isin'], how = 'outer')

In [73]:
df_cleared['cleared_borrowing'].fillna(0, inplace = True)
df_cleared['cleared_lending'].fillna(0, inplace = True)

In [74]:
query = f"""

SELECT business_date, lender_id as entity_id, security_isin, 
CASE
    WHEN s.lender_country_residence IN ('US') THEN 'US'
    WHEN s.lender_country_residence IN ('GB') THEN 'UK'
    WHEN s.lender_country_residence IN
         ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
      THEN 'Euro Area'
    ELSE 'Other'
  END AS residence_group,
CASE
    WHEN contractual_maturity IS NULL OR contractual_maturity <= 7 THEN 'short'
    ELSE 'term'
  END AS tenor,
sum(nominal_value)/1e9 as intra_lending,
sum(repo_rate*nominal_value)/sum(nominal_value) as intra_lending_rate,
avg(contractual_maturity) as intra_lending_maturity,
avg(haircut) as intra_lending_haircut
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND lender_id in {foreign_entity}
AND intragroup = 1
AND central_clearing = 'non-cleared'
AND borrower_country_residence NOT IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
AND repo_rate IS NOT NULL
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.issuer_esa_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.security_isin IN {unique_govs})
  )
GROUP BY business_date, lender_id, residence_group, tenor, security_isin
ORDER BY business_date, lender_id, residence_group, tenor, security_isin
  
"""

df_intra_lending = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_13224\715234798.py:45: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_intra_lending = pd.read_sql_query(query, cnxn)


In [75]:
query = f"""

SELECT business_date, borrower_id as entity_id, security_isin, 
CASE
    WHEN s.borrower_country_residence IN ('US') THEN 'US'
    WHEN s.borrower_country_residence IN ('GB') THEN 'UK'
    WHEN s.borrower_country_residence IN
         ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
      THEN 'Euro Area'
    ELSE 'Other'
  END AS residence_group,
CASE
    WHEN contractual_maturity IS NULL OR contractual_maturity <= 7 THEN 'short'
    ELSE 'term'
  END AS tenor,
sum(nominal_value)/1e9 as intra_borrowing,
sum(repo_rate*nominal_value)/sum(nominal_value) as intra_borrowing_rate,
avg(contractual_maturity) as intra_borrowing_maturity,
avg(haircut) as intra_borrowing_haircut
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND borrower_id in {foreign_entity}
AND intragroup = 1
AND central_clearing = 'non-cleared'
AND lender_country_residence NOT IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
AND repo_rate IS NOT NULL
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.issuer_esa_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.security_isin IN {unique_govs})
  )
GROUP BY business_date, borrower_id, residence_group, tenor, security_isin
ORDER BY business_date, borrower_id, residence_group, tenor, security_isin
  
"""

df_intra_borrowing = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_13224\1751100272.py:45: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_intra_borrowing = pd.read_sql_query(query, cnxn)


In [76]:
df_intra = df_intra_lending.merge(df_intra_borrowing, on = ['business_date', 'entity_id', 'residence_group', 'tenor', 'security_isin'], how = 'outer')

In [77]:
df_intra['intra_lending'].fillna(0, inplace = True)
df_intra['intra_borrowing'].fillna(0, inplace = True)

In [78]:
df = df_cleared.merge(df_intra, on = ['business_date', 'entity_id', 'residence_group', 'tenor', 'security_isin'], how = 'outer')

In [79]:
for c in ['cleared_lending', 'cleared_borrowing', 'intra_lending', 'intra_borrowing']:
    df[c].fillna(0, inplace = True)

In [80]:
df = df[(df['residence_group'] == 'Euro Area') & ((df['intra_lending'] > 0) | (df['intra_borrowing'] > 0))].copy()

In [81]:
df['chain_intra_to_ccp'] = np.minimum(df['intra_borrowing'],
                                df['cleared_lending'])

df['chain_ccp_to_intra'] = np.minimum(df['cleared_borrowing'],
                                df['intra_lending'])

df['wedge_intra_to_ccp'] = (df['cleared_lending_rate'] - df['intra_borrowing_rate'])*100
df['wedge_ccp_to_intra'] = (df['intra_lending_rate'] - df['cleared_borrowing_rate'])*100

df.loc[df['chain_intra_to_ccp'] == 0, 'wedge_intra_to_ccp'] = np.nan
df.loc[df['chain_ccp_to_intra'] == 0, 'wedge_ccp_to_intra'] = np.nan

df['carry'] = (df['wedge_intra_to_ccp'].fillna(0)*df['chain_intra_to_ccp'] + df['wedge_ccp_to_intra'].fillna(0)*df['chain_ccp_to_intra']) / (df['chain_intra_to_ccp'] + df['chain_ccp_to_intra'])

In [82]:
# Market cleared rate of the bond, all banks, volume weighted, by tenor bucket
query = f"""

SELECT business_date, security_isin,
CASE
    WHEN contractual_maturity IS NULL OR contractual_maturity <= 7 THEN 'short'
    ELSE 'term'
  END AS tenor,
sum(nominal_value)/1e9 as market_volume,
sum(repo_rate*nominal_value)/sum(nominal_value) as market_rate
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND intragroup = 0 
AND central_clearing = 'cleared'
AND repo_rate IS NOT NULL
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.issuer_esa_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.security_isin IN {unique_govs})
  )
GROUP BY business_date, security_isin, tenor
ORDER BY business_date, security_isin, tenor
  
"""

df_market = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_13224\1817790719.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_market = pd.read_sql_query(query, cnxn)


In [83]:
df = df.merge(df_market, on = ['business_date', 'security_isin', 'tenor'], how = 'left')

In [85]:
estr = pd.read_csv('Data\\ESTR.csv')

In [86]:
estr['Effective Date'] = pd.to_datetime(estr['DATE'], format='%d/%m/%Y')

In [87]:
df['business_date'] = pd.to_datetime(df['business_date'])

In [88]:
df = df.merge(estr[['Effective Date', 'ESTR']], left_on=['business_date'], right_on = ['Effective Date'], how = 'left')

In [89]:
df['special'] = (df['ESTR'] - df['market_rate'])*100

In [90]:
# Group id and number of entities and groups running a chain in the bond that day
df = df.merge(df_mapping_cc.rename(columns = {'lei': 'entity_id', 'group_id_synth': 'group_id'}), on = 'entity_id', how = 'left')

In [91]:
active = df[(df['chain_intra_to_ccp'] > 0) | (df['chain_ccp_to_intra'] > 0)]
counts = active.groupby(['business_date', 'security_isin'], as_index = False).agg(n_entities = ('entity_id', 'nunique'), n_groups = ('group_id', 'nunique'))
df = df.merge(counts, on = ['business_date', 'security_isin'], how = 'left')
df['n_entities'].fillna(0, inplace = True)
df['n_groups'].fillna(0, inplace = True)

In [92]:
df.to_csv('Data\\wedge_eur.csv', index=False)

# overnight

Same code with `contractual_maturity <= 1` in every query, so both legs are priced on the day and share the tenor. Saves `wedge_eur_on.csv`.

In [93]:
query = f"""

SELECT business_date, lender_id as entity_id, security_isin, 
CASE
    WHEN s.lender_country_residence IN ('US') THEN 'US'
    WHEN s.lender_country_residence IN ('GB') THEN 'UK'
    WHEN s.lender_country_residence IN
         ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
      THEN 'Euro Area'
    ELSE 'Other'
  END AS residence_group,
CASE
    WHEN contractual_maturity IS NULL OR contractual_maturity <= 7 THEN 'short'
    ELSE 'term'
  END AS tenor,
sum(nominal_value)/1e9 as cleared_lending,
sum(repo_rate*nominal_value)/sum(nominal_value) as cleared_lending_rate,
avg(contractual_maturity) as cleared_lending_maturity,
avg(haircut) as cleared_lending_haircut
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND lender_id in {foreign_entity}
AND intragroup = 0
AND central_clearing = 'cleared'
AND contractual_maturity <= 1
AND repo_rate IS NOT NULL
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.issuer_esa_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.security_isin IN {unique_govs})
  )
GROUP BY business_date, lender_id, residence_group, tenor, security_isin
ORDER BY business_date, lender_id, residence_group, tenor, security_isin
  
"""

df_cleared_lending = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_13224\702925185.py:45: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_cleared_lending = pd.read_sql_query(query, cnxn)


In [94]:
query = f"""

SELECT business_date, borrower_id as entity_id, security_isin, 
CASE
    WHEN s.borrower_country_residence IN ('US') THEN 'US'
    WHEN s.borrower_country_residence IN ('GB') THEN 'UK'
    WHEN s.borrower_country_residence IN
         ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
      THEN 'Euro Area'
    ELSE 'Other'
  END AS residence_group,
CASE
    WHEN contractual_maturity IS NULL OR contractual_maturity <= 7 THEN 'short'
    ELSE 'term'
  END AS tenor,
sum(nominal_value)/1e9 as cleared_borrowing,
sum(repo_rate*nominal_value)/sum(nominal_value) as cleared_borrowing_rate,
avg(contractual_maturity) as cleared_borrowing_maturity,
avg(haircut) as cleared_borrowing_haircut
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND borrower_id in {foreign_entity}
AND intragroup = 0
AND central_clearing = 'cleared'
AND contractual_maturity <= 1
AND repo_rate IS NOT NULL
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.issuer_esa_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.security_isin IN {unique_govs})
  )
GROUP BY business_date, borrower_id, residence_group, tenor, security_isin
ORDER BY business_date, borrower_id, residence_group, tenor, security_isin
  
"""

df_cleared_borrowing = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_13224\627953632.py:45: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_cleared_borrowing = pd.read_sql_query(query, cnxn)


In [95]:
df_cleared = df_cleared_lending.merge(df_cleared_borrowing, on = ['business_date', 'entity_id', 'residence_group', 'tenor', 'security_isin'], how = 'outer')

In [96]:
df_cleared['cleared_borrowing'].fillna(0, inplace = True)
df_cleared['cleared_lending'].fillna(0, inplace = True)

In [97]:
query = f"""

SELECT business_date, lender_id as entity_id, security_isin, 
CASE
    WHEN s.lender_country_residence IN ('US') THEN 'US'
    WHEN s.lender_country_residence IN ('GB') THEN 'UK'
    WHEN s.lender_country_residence IN
         ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
      THEN 'Euro Area'
    ELSE 'Other'
  END AS residence_group,
CASE
    WHEN contractual_maturity IS NULL OR contractual_maturity <= 7 THEN 'short'
    ELSE 'term'
  END AS tenor,
sum(nominal_value)/1e9 as intra_lending,
sum(repo_rate*nominal_value)/sum(nominal_value) as intra_lending_rate,
avg(contractual_maturity) as intra_lending_maturity,
avg(haircut) as intra_lending_haircut
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND lender_id in {foreign_entity}
AND intragroup = 1
AND central_clearing = 'non-cleared'
AND borrower_country_residence NOT IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
AND contractual_maturity <= 1
AND repo_rate IS NOT NULL
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.issuer_esa_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.security_isin IN {unique_govs})
  )
GROUP BY business_date, lender_id, residence_group, tenor, security_isin
ORDER BY business_date, lender_id, residence_group, tenor, security_isin
  
"""

df_intra_lending = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_13224\936249010.py:46: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_intra_lending = pd.read_sql_query(query, cnxn)


In [98]:
query = f"""

SELECT business_date, borrower_id as entity_id, security_isin, 
CASE
    WHEN s.borrower_country_residence IN ('US') THEN 'US'
    WHEN s.borrower_country_residence IN ('GB') THEN 'UK'
    WHEN s.borrower_country_residence IN
         ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
      THEN 'Euro Area'
    ELSE 'Other'
  END AS residence_group,
CASE
    WHEN contractual_maturity IS NULL OR contractual_maturity <= 7 THEN 'short'
    ELSE 'term'
  END AS tenor,
sum(nominal_value)/1e9 as intra_borrowing,
sum(repo_rate*nominal_value)/sum(nominal_value) as intra_borrowing_rate,
avg(contractual_maturity) as intra_borrowing_maturity,
avg(haircut) as intra_borrowing_haircut
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND borrower_id in {foreign_entity}
AND intragroup = 1
AND central_clearing = 'non-cleared'
AND lender_country_residence NOT IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
AND contractual_maturity <= 1
AND repo_rate IS NOT NULL
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.issuer_esa_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.security_isin IN {unique_govs})
  )
GROUP BY business_date, borrower_id, residence_group, tenor, security_isin
ORDER BY business_date, borrower_id, residence_group, tenor, security_isin
  
"""

df_intra_borrowing = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_13224\233204755.py:46: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_intra_borrowing = pd.read_sql_query(query, cnxn)


In [99]:
df_intra = df_intra_lending.merge(df_intra_borrowing, on = ['business_date', 'entity_id', 'residence_group', 'tenor', 'security_isin'], how = 'outer')

In [100]:
df_intra['intra_lending'].fillna(0, inplace = True)
df_intra['intra_borrowing'].fillna(0, inplace = True)

In [101]:
df = df_cleared.merge(df_intra, on = ['business_date', 'entity_id', 'residence_group', 'tenor', 'security_isin'], how = 'outer')

In [102]:
for c in ['cleared_lending', 'cleared_borrowing', 'intra_lending', 'intra_borrowing']:
    df[c].fillna(0, inplace = True)

In [103]:
df = df[(df['residence_group'] == 'Euro Area') & ((df['intra_lending'] > 0) | (df['intra_borrowing'] > 0))].copy()

In [104]:
df['chain_intra_to_ccp'] = np.minimum(df['intra_borrowing'],
                                df['cleared_lending'])

df['chain_ccp_to_intra'] = np.minimum(df['cleared_borrowing'],
                                df['intra_lending'])

df['wedge_intra_to_ccp'] = (df['cleared_lending_rate'] - df['intra_borrowing_rate'])*100
df['wedge_ccp_to_intra'] = (df['intra_lending_rate'] - df['cleared_borrowing_rate'])*100

df.loc[df['chain_intra_to_ccp'] == 0, 'wedge_intra_to_ccp'] = np.nan
df.loc[df['chain_ccp_to_intra'] == 0, 'wedge_ccp_to_intra'] = np.nan

df['carry'] = (df['wedge_intra_to_ccp'].fillna(0)*df['chain_intra_to_ccp'] + df['wedge_ccp_to_intra'].fillna(0)*df['chain_ccp_to_intra']) / (df['chain_intra_to_ccp'] + df['chain_ccp_to_intra'])

In [105]:
# Market cleared rate of the bond, all banks, volume weighted, by tenor bucket
query = f"""

SELECT business_date, security_isin,
CASE
    WHEN contractual_maturity IS NULL OR contractual_maturity <= 7 THEN 'short'
    ELSE 'term'
  END AS tenor,
sum(nominal_value)/1e9 as market_volume,
sum(repo_rate*nominal_value)/sum(nominal_value) as market_rate
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND intragroup = 0 
AND central_clearing = 'cleared'
AND contractual_maturity <= 1
AND repo_rate IS NOT NULL
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.issuer_esa_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.security_isin IN {unique_govs})
  )
GROUP BY business_date, security_isin, tenor
ORDER BY business_date, security_isin, tenor
  
"""

df_market = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_13224\4046149814.py:35: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_market = pd.read_sql_query(query, cnxn)


In [106]:
df = df.merge(df_market, on = ['business_date', 'security_isin', 'tenor'], how = 'left')

In [107]:
estr = pd.read_csv('Data\\ESTR.csv')

In [108]:
estr['Effective Date'] = pd.to_datetime(estr['DATE'], format='%d/%m/%Y')

In [109]:
df['business_date'] = pd.to_datetime(df['business_date'])

In [110]:
df = df.merge(estr[['Effective Date', 'ESTR']], left_on=['business_date'], right_on = ['Effective Date'], how = 'left')

In [111]:
df['special'] = (df['ESTR'] - df['market_rate'])*100

In [112]:
# Group id and number of entities and groups running a chain in the bond that day
df = df.merge(df_mapping_cc.rename(columns = {'lei': 'entity_id', 'group_id_synth': 'group_id'}), on = 'entity_id', how = 'left')

In [113]:
active = df[(df['chain_intra_to_ccp'] > 0) | (df['chain_ccp_to_intra'] > 0)]
counts = active.groupby(['business_date', 'security_isin'], as_index = False).agg(n_entities = ('entity_id', 'nunique'), n_groups = ('group_id', 'nunique'))
df = df.merge(counts, on = ['business_date', 'security_isin'], how = 'left')
df['n_entities'].fillna(0, inplace = True)
df['n_groups'].fillna(0, inplace = True)

In [114]:
df.to_csv('Data\\wedge_eur_on.csv', index=False)